# RME Process Chatbot — Prototype

Pipeline: **extract** (PyMuPDF, OCR fallback) → **chunk** by process step → **retrieve** (hybrid BM25 + MiniLM/FAISS) → **generate** (local model via Ollama) → **score** against the 36-question eval set.

All paths resolve relative to the `files/` folder. Source PDFs are read from `../processes_pdf/`.

**What is real in this notebook and what is not:**
- Extraction, chunking, retrieval numbers, and the form-number validator: **real, reproducible, run on this machine**.
- Generation numbers: real only once a local model is served (§4). Until then those cells raise rather than print a fake score.

## 0. Setup

In [1]:
import json, sys, time, re, platform
from pathlib import Path
from collections import defaultdict

def _find_files_dir():
    """Locate the 'files' folder by marker files rather than assuming the CWD.
    Lets the notebook run whether Jupyter was launched from files/ or its parent."""
    here = Path.cwd()
    for c in [here, here / "files", *here.parents]:
        if (c / "eval_set.json").exists() and (c / "chunker.py").exists():
            return c.resolve()
    raise RuntimeError("Could not locate the 'files' folder. Launch Jupyter from inside it.")

FILES = _find_files_dir()
PDF_DIR = FILES.parent / "processes_pdf"
sys.path.insert(0, str(FILES))

from chunker import chunk_all, iter_chunks, route_prefixes, doc_code_prefix
from retriever import Retriever
import validator as V

chunks = chunk_all()                                   # files/extracted_raw.json
eval_set = json.load(open(FILES / "eval_set.json", encoding="utf-8"))
print(f"files dir : {FILES}")
print(f"{len(chunks)} chunks from {len({c['filename'] for c in chunks})} docs, {len(eval_set)} eval questions")

files dir : C:\Users\dahab\Downloads\RME Chatbot\files
108 chunks from 9 docs, 36 eval questions


## 1. Extraction — PyMuPDF primary, OCR fallback

`extract_pipeline.py` reads real PDF bytes from `../processes_pdf/` with PyMuPDF (`fitz`). OCR fires **only** when a page's native text is suspiciously short (<20 chars) or low-alpha (<0.4 ratio).

Run it with `python extract_pipeline.py`. Below is the fallback log from the last run — this is the number to watch as the corpus grows to 500–600 docs.

In [2]:
log = json.load(open(FILES / "ocr_fallback_log.json", encoding="utf-8"))
print(f"docs={log['docs']}  pages={log['total_pages']}  fallback_pages={log['fallback_pages']} ({log['fallback_rate']:.1%})")
for p in log["pages"][:12]:
    print(f"  p{p['page']} of {p['file'][:45]:<45} {p['reason']}  chars={p['chars']}")

docs=9  pages=63  fallback_pages=9 (14.3%)
  p1 of PCM01_Customer_Satisfaction_Process_1.pdf     ocr_failed  chars=0
  p1 of PCM02_Branding_for_Construction_Sites_Process ocr_failed  chars=0
  p1 of PCN01_Subcontract_Agreement_Process_1.pdf     ocr_failed  chars=0
  p1 of PQD12_Quality_Plan_Inspection_and_Testing_Pro ocr_failed  chars=0
  p1 of PSE01_RME_SelfExecution_Process_1.pdf         ocr_failed  chars=0
  p1 of PTN01_Project_Initiation_Process.pdf          ocr_failed  chars=0
  p1 of PTN02_Project_Launching_Process.pdf           ocr_failed  chars=0
  p1 of PVMO01_Vendor_selection_and_Bidding_Process.p ocr_failed  chars=0
  p1 of PVMO02_Procurement_Process.pdf                ocr_failed  chars=0


### What the fallback log actually showed

**The premise was half right.** Pages 2–N of every document do carry clean embedded text — PyMuPDF extracts them perfectly, and re-extracting the whole corpus this way reproduced the BM25 baseline *exactly* (96.9%, same single miss). That part is confirmed.

**But page 1 of all 9 documents is image-only** — 0 text characters, ~16 embedded images. It is the scanned signature/approval cover page (prepared by / reviewed by / approved by, plus title and doc code). So the fallback rate is **9/63 pages = 14.3%**, structurally 1 per document, not "near zero".

Does that justify keeping OCR infrastructure? **No — and the reason is content, not rate.** Checking the previous OCR of those covers shows heavily garbled output (`B_CHO4` for `P-CN-01`, `p-op.12` for `P-QD-12`, `coo` for `COO`). No eval question targets cover-page content, and a mangled doc code on the cover is actively worse than nothing since it pollutes the term index. The right call is `--no-ocr`, or skipping page 1 outright.

If `tesseract` is not installed those 9 pages log as `ocr_failed` rather than being OCR'd, and the run continues on native text. That is deliberate — a missing binary must not kill a 600-document run — and retrieval results are byte-identical either way.

## 2. Retrieval — hybrid BM25 + dense

`retriever.py` now runs two channels and fuses them:
- **BM25** (`rank_bm25`) — exact-token channel, keeps form numbers and doc codes findable.
- **Dense** — `all-MiniLM-L6-v2` via sentence-transformers, normalized, in a FAISS `IndexFlatIP` (exact cosine).

Chunk embeddings are cached to `.embed_cache/` keyed by a corpus+model fingerprint, so restarts don't re-encode.

In [3]:
r = Retriever(chunks)          # default: weighted fusion, dense_weight=0.4
print("model:", r.model_name, "| embeddings:", r.embeddings.shape, "| fusion:", r.fusion)

C:\Users\dahab\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8437.60it/s]

model: all-MiniLM-L6-v2 | embeddings: (108, 384) | fusion: weighted


### 2.1 The eval — `retrieval_hit` unchanged from the baseline harness

In [4]:
def retrieval_hit(question_item, top_k=3, search_fn=None):
    if question_item['doc'] == 'none':
        return None  # unanswerable questions have no target doc
    search_fn = search_fn or r.search
    results = search_fn(question_item['question'], top_k)
    retrieved_docs = [x['filename'] for x in results]
    return question_item['doc'] in retrieved_docs

def score(name, search_fn, top_k=3, show_misses=True):
    t0 = time.perf_counter()
    hits = [retrieval_hit(q, top_k, search_fn) for q in eval_set]
    dt = (time.perf_counter() - t0) / len(eval_set) * 1000
    scored = [h for h in hits if h is not None]
    acc = sum(scored) / len(scored)
    print(f"{name:<30} {sum(scored):>2}/{len(scored)} = {acc:>6.1%}   {dt:>5.1f} ms/q")
    if show_misses:
        for q, h in zip(eval_set, hits):
            if h is False:
                print(f"      MISS [{q['id']}] {q['question'][:70]}")
    return acc

def with_fusion(mode):
    """Measure a different fusion on the SAME retriever.

    Constructing a second Retriever would reload the embedding model and
    re-encode the corpus - wasted work, and it makes the notebook depend on
    network access halfway through. Fusion is just the scoring step, so flip
    it in place and restore."""
    def run(q, k):
        prev = r.fusion
        r.fusion = mode
        try:
            return r.search(q, k, route=False)
        finally:
            r.fusion = prev
    return run

print("top-3 document accuracy")
print("-" * 68)
score("BM25 only (baseline)",      lambda q, k: r.search_bm25(q, k))
score("Dense only (MiniLM)",       lambda q, k: r.search_dense(q, k))
score("Hybrid RRF",                with_fusion('rrf'))
score("Hybrid weighted (w=0.4)",   lambda q, k: r.search(q, k, route=False))
score("Hybrid weighted + routing", lambda q, k: r.search(q, k, route=True))

top-3 document accuracy
--------------------------------------------------------------------
BM25 only (baseline)           31/32 =  96.9%     0.3 ms/q
      MISS [PVMO01-1] What is the form number for the Request for Quotation (RFQ)?


Dense only (MiniLM)            30/32 =  93.8%    78.7 ms/q
      MISS [PVMO01-1] What is the form number for the Request for Quotation (RFQ)?
      MISS [PVMO02-2] What is the minimum number of vendor quotations required for a commerc


Hybrid RRF                     31/32 =  96.9%    15.2 ms/q
      MISS [PVMO01-1] What is the form number for the Request for Quotation (RFQ)?


Hybrid weighted (w=0.4)        32/32 = 100.0%    15.4 ms/q


Hybrid weighted + routing      32/32 = 100.0%    15.1 ms/q


1.0

### 2.2 Result vs. the BM25 baseline

| retriever | top-3 | vs baseline | latency* |
|---|---|---|---|
| BM25 only (baseline) | **96.9%** (31/32) | — | ~1 ms |
| Dense only (MiniLM) | 93.8% (30/32) | −3.1 pt | ~110 ms |
| Hybrid RRF | 96.9% (31/32) | ±0 | ~27 ms |
| **Hybrid weighted (w=0.4)** | **100.0%** (32/32) | **+3.1 pt** | ~29 ms |
| **Hybrid weighted + prefix routing** | **100.0%** (32/32) | **+3.1 pt** | ~30 ms |

\* Retrieval latency on the dev machine (Core Ultra 9 285H, CPU). Varies ±2× run to run; the accuracy column is deterministic, the latency column is not.

The hybrid fixes the exact failure predicted in the brief: `PVMO01-1`, the RFQ form-number lookup that pure BM25 lost to cross-document term overlap.

**Read this honestly, though.** +3.1 points *is one question out of 32*. The eval set is saturated — at 96.9% baseline it cannot resolve any difference finer than one question, so this is not statistically meaningful evidence that weighted-sum beats RRF. Two things make me still recommend the change:

- The 100% is a **plateau, not a lucky point**: `dense_weight` ∈ [0.3, 0.5] all give 32/32, and prefix routing gives 32/32 at *every* weight from 0.0 to 0.7. RRF is flat at 96.9% for every `K` ∈ {1, 5, 10, 20, 60, 100} — it genuinely cannot recover that question.
- Dense-only being *worse* (93.8%) is the useful signal: it confirms embeddings alone are the wrong tool here and BM25 must stay in the loop, exactly as the brief predicted.

**Caveat for scale:** weighted-sum mixes an unbounded BM25 score with a bounded cosine, so its weight may need retuning as the corpus grows 60×. RRF fuses *ranks* and has no such knob. If a larger eval set ever ties them, switch to `Retriever(chunks, fusion='rrf')`.

**To actually measure retrieval improvement you need harder questions** — paraphrased queries that don't share vocabulary with the source ("who signs off on subcontracts?" rather than "who approves the SC Agreement?"). That's where hybrid should pull genuinely ahead, and this eval set contains none of them.

### 2.3 Prefix routing (scale prep)

Pre-retrieval narrowing by doc-code family (PSE/PQD/PCM/PTN/PVMO/PCN) before vector search. On this eval set it fires on 22/36 questions with **0 unsafe routes** — it never excluded the gold document.

In [5]:
fired = unsafe = 0
by_file = defaultdict(set)
for c in chunks:
    by_file[c['filename']].add(doc_code_prefix(c))

for it in eval_set:
    pref = route_prefixes(it['question'])
    if not pref:
        continue
    fired += 1
    if it['doc'] == 'none':
        continue
    if not (by_file[it['doc']] & set(pref)):
        unsafe += 1
        print(f"  UNSAFE [{it['id']}] routed to {pref}, gold is {by_file[it['doc']]}")

print(f"routing fired on {fired}/{len(eval_set)} questions, unsafe routes: {unsafe}")
print("\nNOTE: chunker.PREFIX_HINTS is a hand-written keyword table. Fine for 6 families /")
print("9 docs; at 500-600 docs derive it from document titles or it silently loses recall.")

routing fired on 22/36 questions, unsafe routes: 0

NOTE: chunker.PREFIX_HINTS is a hand-written keyword table. Fine for 6 families /
9 docs; at 500-600 docs derive it from document titles or it silently loses recall.


### 2.4 Scaling `chunk_all()`

`chunker.iter_chunks()` is a generator that yields chunks one document at a time, so an index build never holds the whole corpus. `chunk_all()` remains the eager wrapper this notebook uses. Embeddings cache to disk keyed by corpus fingerprint.

At 600 docs expect ~7k chunks — FAISS `IndexFlatIP` is still exact and sub-millisecond there; only move to IVF/HNSW past ~100k chunks.

In [6]:
t0 = time.perf_counter()
n = sum(1 for _ in iter_chunks())
print(f"streamed {n} chunks without materialising the corpus in {time.perf_counter()-t0:.2f}s")

streamed 108 chunks without materialising the corpus in 0.01s


## 3. Form-number validator (hallucination proxy)

Deterministic, no model in the loop. Harvest every real form number from the corpus, then check what the model cites against it.

Two calibration findings from the actual data:
- The whitelist is built from **all chunks, not just RELATED DOCUMENTED INFORMATION**. The corpus has 67 distinct form numbers; 63 are in RDI sections and **4 appear only in step bodies** (`F-P-HR-02-09`, `F-P-QD-08-04`, `F-P-QP-06-02`, `F-P-VMO-02-01`). An RDI-only whitelist would flag those four *real* forms as hallucinations.
- Policy is **flag, not block**. The corpus cites form families that aren't in it (FW, HR, OP, PU, QP), so at 9 docs "not in whitelist" means *unverifiable*, not *fabricated*. Switch to `policy="block"` once all 500–600 docs are indexed.

In [7]:
FORM_WHITELIST = V.build_form_whitelist(chunks)
DOC_WHITELIST = V.build_doc_code_whitelist(chunks)
print(f"{len(FORM_WHITELIST)} known form numbers, {len(DOC_WHITELIST)} known doc codes")

for probe in [
    "The Customer Satisfaction Survey uses form F-P-CM-01-01.",
    "Use form F-P-CM-03-07 to request a company car.",      # invented
    "Submit F-P-VMO-01-01 and also F-P-ZZ-99-99.",          # one real, one invented
    "Not specified in these process documents.",
]:
    v = V.validate_answer(probe, FORM_WHITELIST, DOC_WHITELIST)
    print(f"  {v['verdict']:<8} cited={str(v['cited']):<32} unknown={v['unknown']}")

67 known form numbers, 9 known doc codes
  clean    cited=['F-P-CM-01-01']                 unknown=[]
  flagged  cited=['F-P-CM-03-07']                 unknown=['F-P-CM-03-07']
  flagged  cited=['F-P-VMO-01-01', 'F-P-ZZ-99-99'] unknown=['F-P-ZZ-99-99']
  clean    cited=[]                               unknown=[]


## 4. Local model serving (Ollama)

Three candidate models. Tags verified against `registry.ollama.ai`:

| model | tag | download |
|---|---|---|
| Llama 3.2 | `llama3.2:latest` (= 3B instruct q4_K_M) | 1.88 GB |
| Gemma 4 E4B | `gemma4:e4b` | 8.95 GB |
| Qwen 3 14B | `qwen3:14b` | 8.64 GB |

```
ollama pull llama3.2:latest
ollama pull gemma4:e4b
ollama pull qwen3:14b
```

Note this is a **wide capability spread** — a 3B model against two ~9 GB models. Expect Llama 3.2 to lose on accuracy and win decisively on latency. That makes the §6 recommendation a real tradeoff rather than a formality.

### Reasoning traces

Qwen 3 is a hybrid-reasoning model and emits `<think>…</think>` blocks by default. That breaks this eval in two ways at once:
- `score_answer` substring-matches the whole response, so a trace musing *"it might be F-P-CM-01-01 or F-P-CN-01-11"* scores as **correct** on keyword match;
- the validator would log a form citation the model never actually made.

Both of the numbers this eval exists to produce would be corrupted. So generation requests set `think: False`, and `strip_reasoning()` removes any trace that slips through before scoring. The raw response is retained in the results for audit.

**Latency across a reasoning and a non-reasoning model is not directly comparable** — §6 notes this rather than ranking them naively.

In [8]:
OLLAMA_HOST = "http://localhost:11434"
MODELS = ["llama3.2:latest", "gemma4:e4b", "qwen3:14b"]

THINK_RE = re.compile(r"<think>.*?</think>", re.S | re.I)

def strip_reasoning(text: str) -> str:
    """Remove <think>...</think> traces before scoring/validation."""
    out = THINK_RE.sub("", text or "")
    # An unterminated trace (hit the token limit mid-think) leaves a dangling
    # open tag; drop everything after it rather than scoring reasoning as answer.
    if "<think>" in out.lower():
        out = re.split(r"<think>", out, flags=re.I)[0]
    return out.strip()

def ollama_up():
    import requests
    try:
        return requests.get(f"{OLLAMA_HOST}/api/tags", timeout=3).status_code == 200
    except Exception:
        return False

def make_generate_REAL(model: str, temperature: float = 0.0, num_predict: int = 512):
    """Return a generate_fn bound to one model. Deterministic (temp=0, fixed seed)
    so the comparison measures the model, not sampling luck."""
    import requests
    def generate_REAL(prompt: str) -> str:
        payload = {
            "model": model, "prompt": prompt, "stream": False,
            "think": False,   # suppress reasoning traces at the source where supported
            "options": {"temperature": temperature, "num_predict": num_predict, "seed": 0},
        }
        resp = requests.post(f"{OLLAMA_HOST}/api/generate", json=payload, timeout=900)
        if resp.status_code == 400:   # older Ollama, or model rejects `think`
            payload.pop("think")
            resp = requests.post(f"{OLLAMA_HOST}/api/generate", json=payload, timeout=900)
        resp.raise_for_status()
        return resp.json()["response"]
    generate_REAL.__name__ = f"generate_REAL[{model}]"
    return generate_REAL

print("Ollama reachable:", ollama_up())

Ollama reachable: False


### 4.1 Record the hardware

The brief asks for *"latency per query on the actual hardware being used"*. That number is meaningless without knowing the machine, so capture it alongside the results.

In [9]:
def hardware_info():
    info = {
        "platform": platform.platform(),
        "processor": platform.processor() or platform.machine(),
        "python": platform.python_version(),
    }
    try:
        import requests
        v = requests.get(f"{OLLAMA_HOST}/api/version", timeout=3).json()
        info["ollama_version"] = v.get("version")
    except Exception:
        info["ollama_version"] = None
    try:
        import torch
        info["cuda"] = torch.cuda.is_available()
        info["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    except Exception:
        info["cuda"], info["gpu"] = None, None
    return info

HW = hardware_info()
for k, v in HW.items():
    print(f"  {k:<16} {v}")
print("\nIf the GPU is not detected above, note it manually - `ollama ps` shows whether")
print("a model is resident on GPU or CPU, and that determines how to read the latency column.")

  platform         Windows-11-10.0.26200-SP0
  processor        Intel64 Family 6 Model 197 Stepping 2, GenuineIntel
  python           3.14.4
  ollama_version   None
  cuda             False
  gpu              None

If the GPU is not detected above, note it manually - `ollama ps` shows whether
a model is resident on GPU or CPU, and that determines how to read the latency column.


## 5. Generation eval harness

`build_prompt` and `score_answer` are unchanged from the baseline. `run_eval` now also records **latency**, strips reasoning traces, and runs the **validator** on every answer. Retrieved context is frozen once so all three models see **byte-identical context** per question.

In [10]:
def build_prompt(question, retrieved_chunks):
    context = '\n\n'.join(f"[{c['filename']} | {c['section']}]\n{c['text']}" for c in retrieved_chunks)
    return (
        "Answer using ONLY the context below. If the answer isn't in the context, "
        "say 'Not specified in these process documents.' Always cite the doc filename.\n\n"
        f"CONTEXT:\n{context}\n\nQUESTION: {question}\nANSWER:"
    )

def score_answer(item, answer_text):
    answer_lower = answer_text.lower()
    return any(kw.lower() in answer_lower for kw in item['must_include'])

# Freeze retrieval once so model comparison is not confounded by retrieval variance.
FROZEN_CONTEXT = {it['id']: r.search(it['question'], 3) for it in eval_set}
FROZEN_PROMPT  = {it['id']: build_prompt(it['question'], FROZEN_CONTEXT[it['id']]) for it in eval_set}
print(f"froze retrieved context for {len(FROZEN_PROMPT)} questions")

def run_eval(generate_fn, top_k=3):
    results = []
    for item in eval_set:
        prompt = FROZEN_PROMPT[item['id']]
        t0 = time.perf_counter()
        raw = generate_fn(prompt)
        latency = time.perf_counter() - t0
        answer = strip_reasoning(raw)          # scored/validated on the stripped answer
        v = V.validate_answer(answer, FORM_WHITELIST, DOC_WHITELIST)
        results.append({
            'id': item['id'], 'question': item['question'], 'type': item['type'],
            'expected': item['expected_answer'],
            'got': answer, 'raw': raw,
            'had_reasoning': raw.strip() != answer,
            'correct': score_answer(item, answer),
            'latency_s': latency,
            'validator_ok': v['ok'], 'cited_forms': v['cited'], 'unknown_forms': v['unknown'],
        })
    return results

froze retrieved context for 36 questions


In [11]:
def report(name, results):
    n = len(results)
    acc = sum(r['correct'] for r in results) / n
    lat = sorted(r['latency_s'] for r in results)
    n_think = sum(r['had_reasoning'] for r in results)
    print(f"\n{'='*64}\n{name}\n{'='*64}")
    print(f"overall accuracy : {sum(r['correct'] for r in results)}/{n} = {acc:.1%}")
    print(f"latency          : mean {sum(lat)/n:.2f}s | median {lat[n//2]:.2f}s | p90 {lat[int(n*0.9)]:.2f}s")
    if n_think:
        print(f"reasoning traces : stripped from {n_think}/{n} answers "
              f"(latency includes the reasoning tokens - not comparable to a non-reasoning model)")

    by_type = defaultdict(list)
    for r in results:
        by_type[r['type']].append(r['correct'])
    print("\naccuracy by type:")
    for t in ['form_lookup', 'fact', 'role_lookup', 'definition', 'fact_crossdoc', 'unanswerable']:
        if t not in by_type: continue
        v = by_type[t]
        star = "   <-- GATING CATEGORY" if t == 'unanswerable' else ""
        print(f"  {t:<16} {sum(v)}/{len(v)} = {sum(v)/len(v):>6.1%}{star}")

    vr = V.validator_report(results, FORM_WHITELIST)
    print("\nvalidator (hallucination proxy, independent of keyword scoring):")
    print(f"  answers citing a form number : {vr['answers_citing_a_form']}/{n}")
    print(f"  answers CAUGHT (unknown form): {vr['answers_with_unknown_form']}/{n} = {vr['caught_rate']:.1%}")
    if vr['unknown_forms']:
        print(f"  invented form numbers        : {', '.join(vr['unknown_forms'])}")
    if vr['fabricated_on_unanswerable']:
        print(f"  !! cited a form on an UNANSWERABLE question: {vr['fabricated_on_unanswerable']}")
    return {'name': name, 'accuracy': acc,
            'by_type': {t: sum(v)/len(v) for t, v in by_type.items()},
            'mean_latency': sum(lat)/n, 'median_latency': lat[n//2],
            'caught_rate': vr['caught_rate'], 'n_reasoning_stripped': n_think}

### 5.1 Run all three models

Runs one model at a time against identical frozen context.

In [12]:
ALL_RESULTS = {}
summaries = []

if not ollama_up():
    raise SystemExit(
        "Ollama is not running. Install it, pull the three models (see \u00a74), "
        "start `ollama serve`, then re-run this cell. "
        "Refusing to print placeholder numbers."
    )

for model in MODELS:
    print(f"\n>>> running {model} ...")
    gen = make_generate_REAL(model)
    res = run_eval(gen)
    ALL_RESULTS[model] = res
    summaries.append(report(model, res))

SystemExit: Ollama is not running. Install it, pull the three models (see §4), start `ollama serve`, then re-run this cell. Refusing to print placeholder numbers.

C:\Users\dahab\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [13]:
# Side-by-side comparison
if summaries:
    print(f"{'model':<24}{'overall':>9}{'unansw.':>9}{'form':>8}{'lat(s)':>9}{'caught':>9}")
    print("-" * 68)
    for s in summaries:
        print(f"{s['name']:<24}{s['accuracy']:>8.1%}{s['by_type'].get('unanswerable',0):>9.1%}"
              f"{s['by_type'].get('form_lookup',0):>8.1%}{s['mean_latency']:>9.2f}{s['caught_rate']:>9.1%}")

### 5.2 Export results

Writes the full run to `model_eval_results.json` so the comparison can be analysed without re-running ~19 GB of models. **This is the file to send back.**

In [14]:
if summaries:
    out = {
        "hardware": HW,
        "models": MODELS,
        "retrieval": {"fusion": r.fusion, "dense_weight": r.dense_weight,
                      "model": r.model_name, "top_k": 3},
        "n_questions": len(eval_set),
        "summaries": summaries,
        "results": ALL_RESULTS,
    }
    dest = FILES / "model_eval_results.json"
    dest.write_text(json.dumps(out, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"wrote {dest}  ({dest.stat().st_size/1024:.0f} KB)")
    print("Send this file back along with the executed notebook.")
else:
    print("No results yet - run \u00a75.1 first.")

No results yet - run §5.1 first.


## 6. Recommendation

**Not yet written — it requires the §5 numbers, which require Ollama.**

Filling this in with anything before those cells run would be fabrication. The decision rule is fixed in advance so the numbers can't be rationalised after the fact:

1. **Disqualify on `unanswerable`.** Any model below ~80% on the unanswerable category is out regardless of overall accuracy. In a compliance chatbot, confidently inventing a form number is worse than answering nothing — a wrong `F-P-…` sends someone to a form that doesn't exist, and the whole point of this tool is that people trust it.
2. **Then disqualify on validator `caught_rate`.** This is the keyword-independent hallucination measure. A model that scores well on `must_include` while inventing *additional* form numbers is failing in a way §5's accuracy column cannot see.
3. **Only then compare overall accuracy**, and treat gaps under ~6% (2 questions) as noise on a 36-question set.
4. **Latency is a tiebreaker, not a criterion** — unless it exceeds ~10 s/query, at which point it's a UX failure.

Two things to hold in mind when reading the output:

- **The size spread is deliberate and matters.** Llama 3.2 (3B) against Gemma 4 E4B and Qwen 3 14B is not a fair fight on accuracy, and it isn't meant to be. If the 3B model clears the `unanswerable` gate and lands within noise of the larger two, it wins on deployability — that is the realistic outcome for a corpus this small and this structured, where retrieval is already doing the hard work at 100% top-3.
- **Qwen 3's latency is not comparable to the others'.** It spends tokens on reasoning before answering. Compare it on accuracy and hallucination rate; treat its latency as a separate question about whether reasoning is worth paying for here.

**A caveat that will apply to whatever the numbers say:** 36 questions across 5 categories means `unanswerable` is decided by **3 questions** and `definition` by 5. A model can go from 100% to 67% on unanswerable by missing one. Treat the §5 output as a screening pass, not a verdict, and expand the eval set — particularly the unanswerable and adversarial categories — before committing to a production model.